# Training Curves Analysis

Анализ кривых обучения для трех архитектур:
- Baseline (EfficientNet-B2)
- StreetCLIP
- GeoCLIP

Визуализация:
- Loss curves (train/val)
- Accuracy curves (top-1, top-3, top-5)
- Haversine distance over epochs
- GeoScore progression
- Learning rate schedule

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

# Настройки визуализации
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

%matplotlib inline

## 1. Загрузка данных обучения

Загружаем логи из W&B или локальных JSON-файлов

In [ ]:
def load_training_history(results_dir: str) -> dict:
    """
    Загружает историю обучения из директории результатов.
    
    Args:
        results_dir: Путь к директории с результатами (например, 'results/baseline')
    
    Returns:
        Dict с историей обучения
    """
    results_dir = Path(results_dir)
    
    # Пытаемся загрузить из training_history.json
    history_file = results_dir / 'training_history.json'
    if history_file.exists():
        with open(history_file, 'r') as f:
            history = json.load(f)
        print(f"Loaded training history from {history_file}")
        return history
    
    # Альтернативно - из metrics.json (если есть)
    metrics_file = results_dir / 'metrics.json'
    if metrics_file.exists():
        with open(metrics_file, 'r') as f:
            metrics = json.load(f)
        print(f"Loaded metrics from {metrics_file}")
        return metrics
    
    raise FileNotFoundError(
        f"No training history found in {results_dir}. "
        "Expected 'training_history.json' or 'metrics.json'"
    )


# Загрузка данных для всех моделей
models = {}

try:
    models['Baseline'] = load_training_history('../results/baseline')
except FileNotFoundError as e:
    print(f"⚠️ Baseline: {e}")

try:
    models['StreetCLIP'] = load_training_history('../results/streetclip')
except FileNotFoundError as e:
    print(f"⚠️ StreetCLIP: {e}")

try:
    models['GeoCLIP'] = load_training_history('../results/geoclip')
except FileNotFoundError as e:
    print(f"⚠️ GeoCLIP: {e}")

if not models:
    print("\n❌ No training results found. Please train models first.")
    print("Run: python code/train.py --config configs/baseline.yaml")
else:
    print(f"\n✅ Loaded {len(models)} model histories")

## 2. Loss Curves

In [ ]:
def plot_loss_curves(models: dict):
    """
    Визуализирует кривые loss для всех моделей.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    colors = {'Baseline': '#2E86AB', 'StreetCLIP': '#A23B72', 'GeoCLIP': '#F18F01'}
    
    # Train loss
    ax = axes[0]
    for name, history in models.items():
        if 'train_loss' in history:
            epochs = range(1, len(history['train_loss']) + 1)
            ax.plot(epochs, history['train_loss'], 
                   label=name, color=colors.get(name, 'gray'), linewidth=2)
    
    ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax.set_ylabel('Training Loss', fontsize=12, fontweight='bold')
    ax.set_title('Training Loss Curves', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # Validation loss
    ax = axes[1]
    for name, history in models.items():
        if 'val_loss' in history:
            epochs = range(1, len(history['val_loss']) + 1)
            ax.plot(epochs, history['val_loss'], 
                   label=name, color=colors.get(name, 'gray'), linewidth=2)
    
    ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax.set_ylabel('Validation Loss', fontsize=12, fontweight='bold')
    ax.set_title('Validation Loss Curves', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/plots/loss_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

if models:
    Path('../results/plots').mkdir(parents=True, exist_ok=True)
    plot_loss_curves(models)

## 3. Accuracy Curves (Top-1, Top-3, Top-5)

In [ ]:
def plot_accuracy_curves(models: dict):
    """
    Визуализирует кривые точности (top-1, top-3, top-5).
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    colors = {'Baseline': '#2E86AB', 'StreetCLIP': '#A23B72', 'GeoCLIP': '#F18F01'}
    metrics = ['top1_acc', 'top3_acc', 'top5_acc']
    titles = ['Top-1 Accuracy', 'Top-3 Accuracy', 'Top-5 Accuracy']
    
    for idx, (metric, title) in enumerate(zip(metrics, titles)):
        ax = axes[idx]
        
        for name, history in models.items():
            val_metric = f'val_{metric}'
            if val_metric in history:
                epochs = range(1, len(history[val_metric]) + 1)
                ax.plot(epochs, history[val_metric], 
                       label=name, color=colors.get(name, 'gray'), 
                       linewidth=2, marker='o', markersize=4)
        
        ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
        ax.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
        ax.set_title(title, fontsize=14, fontweight='bold')
        ax.legend(loc='best', fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig('../results/plots/accuracy_curves.png', dpi=300, bbox_inches='tight')
    plt.show()

if models:
    plot_accuracy_curves(models)

## 4. Geospatial Metrics (Haversine Distance, GeoScore)

In [ ]:
def plot_geospatial_metrics(models: dict):
    """
    Визуализирует геопространственные метрики.
    """
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    colors = {'Baseline': '#2E86AB', 'StreetCLIP': '#A23B72', 'GeoCLIP': '#F18F01'}
    
    # Haversine distance
    ax = axes[0]
    for name, history in models.items():
        if 'val_haversine_km' in history:
            epochs = range(1, len(history['val_haversine_km']) + 1)
            ax.plot(epochs, history['val_haversine_km'], 
                   label=name, color=colors.get(name, 'gray'), 
                   linewidth=2, marker='s', markersize=4)
    
    ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax.set_ylabel('Haversine Distance (km)', fontsize=12, fontweight='bold')
    ax.set_title('Mean Geolocation Error', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    # GeoScore
    ax = axes[1]
    for name, history in models.items():
        if 'val_geoscore' in history:
            epochs = range(1, len(history['val_geoscore']) + 1)
            ax.plot(epochs, history['val_geoscore'], 
                   label=name, color=colors.get(name, 'gray'), 
                   linewidth=2, marker='d', markersize=4)
    
    ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax.set_ylabel('GeoScore', fontsize=12, fontweight='bold')
    ax.set_title('GeoScore (GeoGuessr Formula)', fontsize=14, fontweight='bold')
    ax.legend(loc='best', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/plots/geospatial_metrics.png', dpi=300, bbox_inches='tight')
    plt.show()

if models:
    plot_geospatial_metrics(models)

## 5. Learning Rate Schedule

In [ ]:
def plot_learning_rate(models: dict):
    """
    Визуализирует расписание learning rate.
    """
    fig, ax = plt.subplots(figsize=(12, 6))
    
    colors = {'Baseline': '#2E86AB', 'StreetCLIP': '#A23B72', 'GeoCLIP': '#F18F01'}
    
    for name, history in models.items():
        if 'learning_rate' in history:
            epochs = range(1, len(history['learning_rate']) + 1)
            ax.plot(epochs, history['learning_rate'], 
                   label=name, color=colors.get(name, 'gray'), 
                   linewidth=2.5, alpha=0.8)
    
    ax.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax.set_ylabel('Learning Rate', fontsize=12, fontweight='bold')
    ax.set_title('Learning Rate Schedule (CosineAnnealing)', fontsize=14, fontweight='bold')
    ax.set_yscale('log')
    ax.legend(loc='best', fontsize=11)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig('../results/plots/learning_rate.png', dpi=300, bbox_inches='tight')
    plt.show()

if models:
    plot_learning_rate(models)

## 6. Comparison Summary Table

In [ ]:
def create_comparison_table(models: dict) -> pd.DataFrame:
    """
    Создает сравнительную таблицу финальных метрик.
    """
    rows = []
    
    for name, history in models.items():
        row = {'Model': name}
        
        # Берем последние значения метрик
        metrics_to_extract = [
            ('val_loss', 'Val Loss'),
            ('val_top1_acc', 'Top-1 Acc'),
            ('val_top3_acc', 'Top-3 Acc'),
            ('val_top5_acc', 'Top-5 Acc'),
            ('val_haversine_km', 'Mean Distance (km)'),
            ('val_geoscore', 'GeoScore'),
        ]
        
        for metric_key, metric_name in metrics_to_extract:
            if metric_key in history:
                value = history[metric_key][-1]  # Последнее значение
                
                # Форматирование
                if 'acc' in metric_key:
                    row[metric_name] = f"{value:.3f}"
                elif 'distance' in metric_key or 'haversine' in metric_key:
                    row[metric_name] = f"{value:.1f}"
                elif 'geoscore' in metric_key:
                    row[metric_name] = f"{value:.1f}"
                else:
                    row[metric_name] = f"{value:.4f}"
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df

if models:
    comparison_df = create_comparison_table(models)
    print("\n📊 Final Metrics Comparison:\n")
    print(comparison_df.to_string(index=False))
    
    # Сохранение в CSV для thesis
    comparison_df.to_csv('../results/model_comparison.csv', index=False)
    print("\n✅ Saved to: results/model_comparison.csv")

## 7. Export Plots for Thesis

Все графики сохранены в `results/plots/` в высоком разрешении (300 DPI) для использования в дипломе.

In [ ]:
print("\n✅ All training curves analyzed and saved!")
print("\nGenerated files:")
print("  - results/plots/loss_curves.png")
print("  - results/plots/accuracy_curves.png")
print("  - results/plots/geospatial_metrics.png")
print("  - results/plots/learning_rate.png")
print("  - results/model_comparison.csv")